# 대한민국 법률 서비스 접근성 불균형 분석
## 지역별 종합 법률 접근성 지수(LLAI) — 분석 정리 노트북

서강대 26-1 빅데이터컴퓨팅(BDS3010) · 20210717 김찬우

이 노트북은 `src/` 모듈(수집·정제·지수·모델 **엔진**)을 불러와 전체 분석 흐름을 한눈에 정리한다.  
데이터 처리 로직은 `src/`에 두고, 노트북은 **결과를 재현·설명·해석**한다.

```
src/collect/scrape_bar.py   변협 회원현황 스크랩
src/prepare/load_kosis.py   KOSIS 인구·GRDP·수급자 정제
src/prepare/load_klac.py    공단 법률구조(A3) 정제
src/prepare/load_court.py   사법연감 법원사건(A2) 정제
src/prepare/build_panel.py  13권역·10도단위 패널 조립
src/index/compute_llai.py   A1~A3 -> 정규화 -> 가중치 3종 -> LLAI
src/model/cluster.py        K-means 클러스터링
```

## 1. 배경 · 연구 목적

로스쿨 도입(2009) 이후 변호사 수가 4만 명을 넘어섰지만, 법률 서비스 자원의 **지역 편중**은 심화되고 있다는 지적이 있다. 본 프로젝트는 지역별 법률 접근성을 단일 종합지수(**LLAI**)로 정량화하고 격차의 구조를 규명한다.

**3개 차원**

| 지표 | 정의 | 방향 |
|---|---|---|
| A1 변호사 접근성 | 인구 10만 명당 개업변호사 수 | 높을수록 좋음 |
| A2 사건 부담 | 변호사 1인당 법원 본안사건 수 | 낮을수록 좋음 |
| A3 취약계층 접근성 | 법률구조 건수 / 저소득층 인구 | 높을수록 좋음 |

**가설** — H1: 수도권 변호사 1인당 인구 < 비수도권 / H2: 변호사 적을수록 사건부담↑ / H3: 소득 낮을수록 법률구조 수요↑·자원↓ / H4: 로스쿨 이후에도 격차 미축소

## 2. 데이터 출처 · 분석 단위

| 차원 | 출처 | 단위 | 기간 |
|---|---|---|---|
| A1 변호사 | 대한변호사협회 회원현황(스냅샷) | 14 지방회 | 현재 |
| A2 법원사건 | 대법원 **사법연감 2024**(본안사건) | 법원관내 | 2024 |
| A3 법률구조 | 대한법률구조공단(민사, 지역별) | 10 도단위 | 2012~2025 |
| 인구·저소득층·GRDP | 통계청 KOSIS | 17 시도 | 2008~2025 |

### 분석 단위가 두 가지인 이유
각 출처의 지역 해상도가 다르다. 변협은 14지방회(부산·인천·울산 분리), 공단은 10도단위(광역시를 인접 도에 통합)다. 그래서 **두 단위를 병행**한다.

- **region13** — 변협 기준 13권역. 부산·인천·울산 분리 유지. A3는 인구비 근사배분.
- **region10** — 공단 기준 10도단위. 모든 지표 공통 해상도(가장 정합적).

단일 기준표: `region_mapping.csv`.  
**A2는 '본안사건'**(민사본안+형사공판+가사+행정 제1심) 사용 — 비송(등기·공탁 980만건) 제외.

> 검증: 법원관내 인구 합 = region10 KOSIS 인구와 정확히 일치(부산+울산+창원관내=경남권 759만). 본안사건 합 1,106,526 = 사법연감 전국 본안과 일치.

In [ ]:
# --- 환경 설정 ---
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

import config
from index.compute_llai import compute_llai, NEED, DIRECTED
from model.cluster import choose_k
from sklearn.cluster import KMeans

pd.set_option('display.unicode.east_asian_width', True)
print('프로젝트 루트:', ROOT)
print('LLAI 입력 컬럼:', NEED)

## 3. 데이터 수집·정제 파이프라인

각 `src` 스크립트가 원본을 tidy 형태로 변환한다. (재생성: 터미널에서 `python src/prepare/load_kosis.py` 등) 아래는 정제 산출물 미리보기.

In [ ]:
# 변호사(A1) — 변협 스냅샷 -> 13권역
bar = pd.read_csv(ROOT / 'data/raw/bar/bar_region13.csv')
display(bar)

In [ ]:
# KOSIS 정제 결과 (인구 / 1인당 GRDP / 기초생활수급자)
for name in ['kosis_population', 'kosis_grdp', 'kosis_basic']:
    d = pd.read_csv(ROOT / f'data/interim/{name}.csv')
    print(f'[{name}] {len(d)}행  {d["year"].min()}~{d["year"].max()}')
    display(d.head(3))

In [ ]:
# 법률구조(A3, region10) / 법원사건(A2, region13)
klac = pd.read_csv(ROOT / 'data/interim/klac_legalaid.csv')
court = pd.read_csv(ROOT / 'data/interim/court_cases.csv')
print('A3 법률구조(민사, 연합산) 최신연도:'); display(klac[klac.year == klac.year.max()])
print('A2 법원 본안사건(2024):'); display(court)

## 4. 통합 패널 (region10 / region13)

`build_panel.py`가 5개 원천을 권역×연도 패널로 합친다. 변호사 스냅샷이 있는 **기준연도(2024)** 단면에서 LLAI 입력 5종이 모두 채워진다.

In [ ]:
def cross_section(unit):
    """패널에서 LLAI 입력이 모두 존재하는 최신 연도 단면을 반환."""
    p = pd.read_csv(ROOT / f'data/processed/panel_{unit}.csv')
    ref = int(p.dropna(subset=NEED)['year'].max())
    return p[p['year'] == ref].dropna(subset=NEED).copy(), ref

cs10, ref10 = cross_section('region10')
print(f'region10 기준연도 {ref10} · {len(cs10)}개 권역')
display(cs10[['region'] + NEED])

## 5. LLAI 산출 — 가중치 3종 비교

A1~A3를 Min-Max 정규화(A2는 역방향) 후 **균등 / 엔트로피 / PCA** 세 가중치로 산출한다.  
지표가 3개뿐이라 단일 가중치는 불안정하므로 **강건성** 확인을 위해 병기한다.

In [ ]:
def llai_table(unit):
    cs, ref = cross_section(unit)
    df, w = compute_llai(cs)
    df = df.sort_values('LLAI', ascending=False)
    print(f'=== {unit} (기준연도 {ref}) ===')
    print('가중치(행=방식):'); display(w.round(3))
    cols = ['region', 'A1', 'A2', 'A3', 'LLAI_equal', 'LLAI_entropy', 'LLAI_pca', 'LLAI']
    display(df[cols].round(2))
    return df

llai10 = llai_table('region10')

In [ ]:
llai13 = llai_table('region13')

### 해석
- **서울이 압도적 1위** (LLAI 87~91). 변호사 263명/10만명으로 2위권(13~37)과 격차 극단적.
- 가중치 방식별로 A2 비중이 크게 갈린다(PCA ~0.4 vs 엔트로피 ~0.13) → **가중치 선택이 순위에 영향**.
- 엔트로피 가중치가 A1에 0.64~0.69 부여 → **변호사 분포 불균등이 격차를 지배**(H1 지지).

## 6. 클러스터링 — 접근성 유형별 권역 그룹

방향보정·정규화 지표(A1n, 1−A2n, A3n)로 K-means. 최적 K는 Elbow+Silhouette로 결정.

In [ ]:
def cluster_table(df, unit):
    cur = df.copy()
    X = cur[DIRECTED].to_numpy()
    diag = choose_k(X)
    best_k = int(diag.loc[diag['silhouette'].idxmax(), 'k'])
    km = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(X)
    cur['cluster'] = km.labels_
    order = cur.groupby('cluster')['LLAI'].mean().sort_values(ascending=False)
    rank = {c: i for i, c in enumerate(order.index)}
    cur['cluster_rank'] = cur['cluster'].map(rank)
    print(f'=== {unit} · 최적 K={best_k} ===')
    display(diag.round(3))
    out = cur.sort_values(['cluster_rank', 'LLAI'], ascending=[True, False])
    display(out[['region', 'A1', 'A2', 'A3', 'LLAI', 'cluster_rank']].round(2))
    return out

_ = cluster_table(llai10, 'region10')

In [ ]:
_ = cluster_table(llai13, 'region13')

## 7. 주요 발견

1. **서울 일극 집중** — A1에서 서울이 극단적 이상치. 클러스터링에서 단독 그룹 분리(K=2).
2. **수도권 우위(H1 지지)** — 서울·인천·경기 상위. 비수도권은 사건부담(A2)도 높음.
3. **A3 변별력 약함** — 값이 0.03~0.07로 좁게 분포. 법률구조가 수요를 충분히 반영 못할 가능성(H3).
4. **가중치 민감성** — LLAI 순위가 가중치 방식에 따라 일부 변동 → 단일 지수 해석 시 주의.

## 8. 남은 작업

- [ ] **시각화**: 코로플레스 지도(folium), LLAI 막대/산점도, 클러스터 지도
- [ ] **회귀분석**: 구조요인(1인당 GRDP·고령화율·인구밀도) → LLAI (statsmodels)
- [ ] **가설 검정**: H1~H4 정량 검정
- [ ] **보고서·발표자료**

> 시각화·회귀에는 `pip install matplotlib seaborn folium geopandas statsmodels` 필요.

In [ ]:
# (선택) 시각화 예시 — matplotlib 설치 후 실행
# !pip install matplotlib
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

d = llai13.sort_values('LLAI')
plt.figure(figsize=(8, 5))
plt.barh(d['region'], d['LLAI'], color='#4C72B0')
plt.xlabel('LLAI (0~100)')
plt.title('region13 권역별 종합 법률 접근성 지수 (2024)')
plt.tight_layout(); plt.show()